# Gastric Xenium — end-to-end ASTER-SC spatial domains (Fig. 5)

Runs the whole gastric cancer workflow from the raw Xenium bundle and the
standardized H&E to the K=17 spatial domain map of Fig. 5 (BS06 tumour section,
696,313 cells x the 377-gene panel).

Every model and training loop comes from `repro_st_aster`; this notebook only wires
the stages together. The equivalent command-line path is
`scripts/prepare_gastric_xenium_*.sh` + `scripts/reproduce_gastric_xenium_*.sh`.

## Data you need to download first

This repository ships the data directories **empty**. Download the bundle and
unpack it into the paths below (see `raw_data/gastric_xenium/README.md`):

```text
# 1. Raw Xenium cell_feature_matrix.h5 -> raw_data/gastric_xenium/
#    <GASTRIC_RAW_ST_URL>
# 2. Standardized H&E image + cell_coordinates.csv -> raw_data/gastric_xenium/
#    <GASTRIC_HE_URL>
# 3. Pre-extracted UNI-2 features -> preprocess_data/gastric_xenium/uni/
#    <GASTRIC_UNI_FEATURES_URL>
```

Item 3 is optional if you have your own UNI-2 weights: set `HAVE_UNI2_WEIGHTS = True`
below and the features are extracted on the fly instead. UNI-2 weights are not
redistributed here.

Environment: `conda activate repro_st_aster` (see the top-level README).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

from repro_st_aster.common import find_repo_root, get_palette
from repro_st_aster.uni_bcam.bcam_prepare_inputs import prepare_inputs

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / 'raw_data' / 'gastric_xenium'
PRE_DIR = REPO_ROOT / 'preprocess_data' / 'gastric_xenium'
UNI_DIR = PRE_DIR / 'uni'
BCAM_INPUT_DIR = PRE_DIR / 'bcam_input'
INR_DIR = PRE_DIR / 'inr_output'
BCAM_DIR = PRE_DIR / 'bcam_output'
VIS_DIR = PRE_DIR / 'viz'

MATRIX_H5 = RAW_DIR / 'cell_feature_matrix.h5'
COORD_CSV = RAW_DIR / 'cell_coordinates.csv'
HE_IMAGE = RAW_DIR / 'tissue_standardized_0p5um.jpg'

# Route A extracts UNI-2 features from the H&E (needs the gated weights);
# route B loads the pre-extracted feature grid. Everything downstream is identical.
HAVE_UNI2_WEIGHTS = False
UNI2_WEIGHTS_DIR = RAW_DIR / 'uni2-h'

# Set an integer (e.g. 4000) for a quick smoke run; None reproduces the full figure.
# This is the notebook equivalent of the CLI --max-cells flag.
SUBSET_N = None

for path in [MATRIX_H5, COORD_CSV, UNI_DIR / 'superpixel_features.npy']:
    print(f'{path.exists()!s:>5}  {path}')

## Step 0 — raw cells to model inputs

`prepare_inputs` reads the Xenium single-cell counts, normalizes them
(`normalize_total(1e4)` + `log1p`, all 377 panel genes, no cell QC filtering), and
attaches to each cell the UNI-2 feature of the superpixel it falls in, using
`floor(coord / stride)` on the standardized coordinates.

`superpixel_stride=16` is the value the published Fig. 5 run used and is kept here so
the figure reproduces. Note that the feature grid's true stride is 14 px
(16352 / 1168), so a stride of 16 addresses only the leading part of the grid — see
the README "Deviations" section. One cell (x = -87.5) falls outside the grid and is
dropped, giving 696,313 from 696,314.

In [ ]:
meta = prepare_inputs(
    matrix_h5=MATRIX_H5,
    coord_csv=COORD_CSV,
    uni_feature_path=UNI_DIR / 'superpixel_features.npy',
    out_dir=BCAM_INPUT_DIR,
    max_cells=SUBSET_N,
    superpixel_stride=16,   # published research value; grid stride is really 14 px
)
for key, value in meta.items():
    print(f'  {key:26s} {value}')

## Step 1 — UNI-2 histology features

The standardized H&E (40096 x 16352 px at 0.5 µm/px) is cut into 224x224 tiles;
UNI-2 (ViT-Giant/14) emits 16x16 patch tokens per tile, stitched into a
`(1168, 2864, 1536)` grid. Tile rows are processed in chunks of 32 tiles so that this
wide slide does not exhaust GPU memory.

Route B just checks that the downloaded grid is present — Step 0 already read it to
attach a feature vector to every cell.

In [ ]:
if HAVE_UNI2_WEIGHTS:
    from repro_st_aster.uni_bcam.uni_extract import extract_features

    uni_meta = extract_features(
        image_path=HE_IMAGE,
        model_dir=UNI2_WEIGHTS_DIR,
        out_dir=UNI_DIR,
        superpixel_stride=16,
        tile_batch_size=32,
        write_pickle=False,
    )
    print(uni_meta)
else:
    uni_grid = np.load(UNI_DIR / 'superpixel_features.npy', mmap_mode='r')
    print('pre-extracted UNI-2 grid:', uni_grid.shape)

uni_per_cell = np.load(BCAM_INPUT_DIR / 'uni2_features_per_cell.npy', mmap_mode='r')
print('UNI-2 features per cell: ', uni_per_cell.shape)

## Step 2 — INR + low-rank Tucker-2 reconstruction

`reconstruct` fits the continuous field `f(s, g) = SIREN(s) @ K @ g_emb^T` on the
irregular single-cell point cloud (depth 6, spatial rank 512, gene rank 256, omega
ramp 1 -> 5, weighted Huber loss with zeros down-weighted to 0.1). Coordinates get
0.002 of jitter each epoch, and training early-stops on a 5% validation split with
patience 50 — the published run stopped at epoch 654 of 3000.

In [ ]:
from repro_st_aster.aster_sc import reconstruct

inr_args = [
    '--data-dir', str(BCAM_INPUT_DIR),
    '--out-dir', str(INR_DIR),
    '--batch-size', '2048',
    '--depth', '6',
    '--xy-jitter', '0.002',
    '--epochs', '30' if SUBSET_N else '3000',
    '--patience', '5' if SUBSET_N else '50',
]
if SUBSET_N:
    inr_args += ['--max-cells', str(SUBSET_N)]

reconstruct.main(inr_args)

In [ ]:
inr_expr = np.load(INR_DIR / 'inr_reconstructed_expression.npy', mmap_mode='r')
raw_expr = np.load(BCAM_INPUT_DIR / 'gene_expression_normalized.npy', mmap_mode='r')
coords = np.load(BCAM_INPUT_DIR / 'cell_coords_standardized.npy')
gene_names = np.load(BCAM_INPUT_DIR / 'gene_names.npy', allow_pickle=True)
print('INR reconstruction:', inr_expr.shape)

gene_idx = 0
span = np.ptp(coords, axis=0)
fig, axes = plt.subplots(2, 1, figsize=(16, 2 * 8 * span[1] / span[0] + 1))
for ax, values, title in [
    (axes[0], np.asarray(raw_expr[:, gene_idx]), 'raw lognorm'),
    (axes[1], np.asarray(inr_expr[:, gene_idx]), 'INR reconstruction'),
]:
    ax.scatter(coords[:, 0], coords[:, 1], c=values, s=0.5, cmap='turbo',
               edgecolors='none', rasterized=True)
    ax.set_title(f'{gene_names[gene_idx]} - {title}')
    ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
plt.tight_layout()

## Step 3 — BCAM fusion: INR expression x UNI-2 morphology

`BCAM` (in `repro_st_aster.uni_bcam.bcam_core`) runs three layers of bidirectional
local cross-attention over each cell's K=8 spatial neighbourhood — self excluded
from the graph, then re-added as the query token inside the attention — fuses the
gene and morphology streams into a 512-d latent, and reconstructs the raw expression
through a Softplus head (MSE, 20 epochs, lr 1e-4 with cosine warm restarts).

Note the return order is `(latent, recon)`, the reverse of `FusionNet`.

In [ ]:
from repro_st_aster.aster_sc import bcam

bcam_args = [
    '--data-dir', str(BCAM_INPUT_DIR),
    '--inr-dir', str(INR_DIR),
    '--out-dir', str(BCAM_DIR),
    '--epochs', '3' if SUBSET_N else '20',
    '--batch-size', '512',
    '--k-neighbors', '8',
    '--scheduler', 'cosine_warm_restarts',
]
if SUBSET_N:
    bcam_args += ['--max-cells', str(SUBSET_N)]

bcam.main(bcam_args)

## Step 4 — spatial domains (KMeans K=17)

`KMeans(K=17, random_state=42, n_init=20)` runs on the **unsmoothed** 512-d latent —
BCAM's local cross-attention already pools over each neighbourhood, so no extra
spatial smoothing is applied here (unlike the colorectal workflow).

Domains are coloured with the published Fig. 5 scheme, indexed by domain ID.

In [ ]:
from repro_st_aster.aster_sc import cluster_visualize

cluster_args = [
    '--data-dir', str(BCAM_INPUT_DIR),
    '--bcam-dir', str(BCAM_DIR),
    '--inr-dir', str(INR_DIR),
    '--vis-dir', str(VIS_DIR),
    '--k', '17',
    '--smooth-k', '0',
    '--kmeans-random-state', '42',
    '--kmeans-n-init', '20',
    '--palette', 'fig5_gastric',
    '--point-size', '0.5', '--point-alpha', '0.85', '--invert-yaxis',
]
if SUBSET_N:
    cluster_args += ['--max-cells', str(SUBSET_N)]

cluster_visualize.main(cluster_args)

In [ ]:
labels = np.load(VIS_DIR / 'labels_fusion_K17.npy')
coords = np.load(BCAM_INPUT_DIR / 'cell_coords_standardized.npy')[: len(labels)]

span = np.ptp(coords, axis=0)
fig, ax = plt.subplots(figsize=(18, 18 * span[1] / span[0]))
ax.scatter(coords[:, 0], coords[:, 1], c=labels, cmap=ListedColormap(get_palette('fig5_gastric', 17)),
           s=0.5, alpha=0.85, edgecolors='none', rasterized=True)
ax.set_title(f'Gastric ASTER-SC spatial domains (K=17, {len(labels):,} cells)')
ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
plt.tight_layout()

for domain_id, count in zip(*np.unique(labels, return_counts=True)):
    print(f'  domain {domain_id:2d}: {count:7,d} cells ({count / len(labels) * 100:5.2f}%)')

## Rerunning from the shell

The same pipeline without a notebook:

```bash
bash scripts/prepare_gastric_xenium_uni.sh --model-dir /path/to/uni2-h   # optional, route A only
bash scripts/prepare_gastric_xenium_bcam_input.sh
bash scripts/reproduce_gastric_xenium_inr.sh
bash scripts/reproduce_gastric_xenium_bcam.sh
bash scripts/reproduce_gastric_xenium_cluster.sh
```

Smoke test (minutes rather than hours):

```bash
bash scripts/prepare_gastric_xenium_bcam_input.sh --max-cells 4000
bash scripts/reproduce_gastric_xenium_inr.sh --epochs 30 --patience 5 --max-cells 4000
bash scripts/reproduce_gastric_xenium_bcam.sh --epochs 3 --batch-size 128 --max-cells 4000
bash scripts/reproduce_gastric_xenium_cluster.sh --max-cells 4000
```